# 24 - Pass 11c: CPU analyses on existing predictions (no GPU)

Two analyses on predictions that already exist in the repo (nothing is trained):

1. **Polarity-flip difference-in-differences (H11.1, H11.2).** Do the helper heads hurt sarcastic headlines whose teacher sentiment is *positive* (surface sentiment points the wrong way)
   more than other sarcastic headlines? A2 vs A1 and A31 vs A20, at 1% and 10%; 107 positive-surface sarcastic test headlines vs 1,248 other sarcastic ones. Exploratory (n = 107).
2. **Seed-aware bootstrap (H11.3, H11.4, H11.23).** Each resample draws the five training seeds with replacement and then the test headlines, so the uncertainty from the five labeled
   subsets is included. Reported next to the test-only interval used in every earlier table.

Predictions were written and frozen before this notebook (`vault/Pass 11 Predictions.md`); the notebook refuses to start unless that file says FROZEN.
**Settings:** Accelerator **None (CPU)**, Internet **on**, Input: the `gainsplit-processed` dataset (for `helper_labels.csv`). About 5-10 minutes.

In [ ]:
import os, subprocess, sys, time, json, shutil
from pathlib import Path
T_START = time.time()
REPO_URL = "https://github.com/Ariffurrhmn/GainSplit.git"
REPO_DIR = "/tmp/GainSplit"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")
COMMIT = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
print("code commit:", COMMIT)

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "tests/test_pass11_cpu.py", "-q", "-p", "no:cacheprovider"],
                   cwd=REPO_DIR, capture_output=True, text=True)
print(r.stdout[-1200:], r.stderr[-800:])
assert r.returncode == 0, "tests failed: see output above"

In [ ]:
REQUIRE_FROZEN = True
pred = Path(f"{REPO_DIR}/vault/Pass 11 Predictions.md").read_text(encoding="utf-8")
status = [l.split(":", 1)[1].strip() for l in pred.splitlines()[:8] if l.startswith("status:")]
print("predictions status:", status)
if REQUIRE_FROZEN:
    assert status == ["FROZEN"], "predictions must be FROZEN in this commit"

In [ ]:
from gainsplit.data import find_data_dir, verify
DATA_DIR = find_data_dir("/kaggle/input")
verify(DATA_DIR)
print("data:", DATA_DIR, "- all files match manifest.json")

## Analysis (10,000 resamples per contrast)

In [ ]:
from gainsplit import analysis_pass11_cpu as cpu
import pandas as pd
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 120)
OUT_DIR = "/kaggle/working/results/pass11_cpu"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
DIRS = [f"{REPO_DIR}/results/{p}" for p in ("pass1", "pass2a", "pass2b", "pass8_mlm", "pass8_jepa", "pass9",
                                            "pass10_stack", "pass10_seeds", "pass11")]
print("results folders:", [d for d in DIRS if not Path(d).exists()] or "all found")
tables = cpu.run_all_cpu(DIRS, DATA_DIR, f"{OUT_DIR}/analysis", n=10000)
cl = tables["cluster_bootstrap"]
a = cl[(cl["contrast"] == "A20-A1") & (cl["budget"] == 1)].iloc[0]
assert abs(a["mean_diff"] - 7.42) < 0.02, "sanity check failed: A20 - A1 at 1% should be +7.42 (Pass 8a)"
print("sanity check passed: A20 - A1 at 1% =", a["mean_diff"])
for name in ("polarity_did", "cluster_bootstrap", "verdicts"):
    print(f"\n=== {name}")
    print(tables[name].to_string())

In [ ]:
print(shutil.make_archive("/kaggle/working/results_pass11_cpu", "zip", OUT_DIR))
print(f"notebook time: {(time.time() - T_START) / 60:.1f} min")